# Step 6: Run the five models on Assam and make the flood maps

**Run this on Kaggle.** Settings: Accelerator **GPU T4 x2**, **Internet on**.

**Add two datasets** (right panel, *Add Input*):
- `cutoff-assam-inputs`: the three `.tif` files from Step 5
- `cutoff-models`: the five files `unet_seed0.pth` to `unet_seed4.pth`

**What it does**
1. Reads the before (17 July) and during (29 July) radar images and runs all five models over the whole area, in overlapping 512 x 512 tiles.
2. Applies the flood rule **separately for each model**: water on 29 July, dry on 17 July, not permanent water (JRC above 80%), not high ground (HAND above 15 m).
3. Saves the maps as GeoTIFFs on the same grid, plus a summary table and quick-look pictures.

**Outputs** (in `outputs/`, download from the *Output* tab):

| File | What it holds |
|---|---|
| `prob_post_mean.tif` | Average water probability on 29 July (0 to 250 = 0 to 1; 255 = no data) |
| `prob_post_std.tif` | Uncertainty: spread of the five models on 29 July (value / 500 = std; 255 = no data) |
| `prob_pre_mean.tif` | Average water probability on 17 July (same scale) |
| `flood_bits.tif` | Flood rule, one bit per model: bit k set = model k says flooded |
| `water29_bits.tif` | Same, but **without** the "dry on 17 July" rule (for the open decision on roads already under water) |
| `flood_count.tif` | How many of the five models say flooded (0 to 5) |
| `summary.csv`, `figures/` | Areas in km2 and pictures |

Runtime: roughly 10 to 20 minutes on a T4.

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

In [ ]:
import os, sys, glob, json, time, subprocess
import numpy as np

REPO = "/tmp/cutoff-flood"
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from src import config
from src.preprocess import preprocess_sar

OUT = "/kaggle/working/outputs" if os.path.exists("/kaggle/working") else "/content/outputs"
os.makedirs(f"{OUT}/figures", exist_ok=True)

THRESHOLD = 0.35          # chosen on the validation set in Step 4 (results/kaggle_run1/run_info.json)
NODATA_IN = -32768
TILE = config.TILE_SIZE   # 512
STRIDE = 448              # 64 pixels of overlap between neighbouring tiles
BATCH = 8

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)) or \
           sorted(glob.glob(f"/content/**/{pattern}", recursive=True))
    assert hits, f"Could not find {pattern}. Did you add the dataset?"
    return hits

PRE_TIF = find("s1_pre_20260717*.tif")
POST_TIF = find("s1_post_20260729*.tif")
AUX_TIF = find("aux_jrc_hand*.tif")
MODEL_FILES = find("unet_seed*.pth")
assert len(PRE_TIF) == len(POST_TIF) == len(AUX_TIF) == 1, \
    "Earth Engine split a file into parts; tell Claude, the reading step needs to merge them."
assert len(MODEL_FILES) == config.ENSEMBLE_SIZE, f"Expected 5 models, found {len(MODEL_FILES)}"
print("Inputs found:", PRE_TIF[0], POST_TIF[0], AUX_TIF[0], sep="\n  ")
print("Models:", [os.path.basename(m) for m in MODEL_FILES])

## 1. Read the inputs and check they line up

All three files must have the same grid. Radar values go back to dB (divide by 100); no-data pixels are recorded so they are never called water.

In [ ]:
import rasterio

def read(path):
    with rasterio.open(path) as f:
        return f.read(), f.profile

pre_raw, prof = read(PRE_TIF[0])
post_raw, prof_post = read(POST_TIF[0])
aux_raw, prof_aux = read(AUX_TIF[0])
for p in (prof_post, prof_aux):
    assert p["transform"] == prof["transform"] and p["crs"] == prof["crs"] \
        and (p["height"], p["width"]) == (prof["height"], prof["width"]), "Grids do not match"
H, W = prof["height"], prof["width"]
print(f"Grid OK: {W} x {H} pixels, {prof['crs']}")

def to_db(raw):
    nod = (raw == NODATA_IN).any(axis=0)
    db = raw.astype(np.float32) / 100.0
    db[:, nod] = np.nan
    return db, nod

pre_db, pre_nod = to_db(pre_raw); del pre_raw
post_db, post_nod = to_db(post_raw); del post_raw
jrc = aux_raw[0].astype(np.int16)
hand_m = np.where(aux_raw[1] == NODATA_IN, np.nan, aux_raw[1] / 10.0).astype(np.float32)
del aux_raw
valid = ~pre_nod & ~post_nod
print(f"Pixels with both radar images: {valid.mean():.1%}")

# Sanity check: typical land is about -15 to -5 dB (VV), water below about -18 dB.
for name, db in [("17 July", pre_db), ("29 July", post_db)]:
    q = np.nanpercentile(db[:, ::7, ::7].reshape(2, -1), [1, 50, 99], axis=1)
    print(f"{name}: VV 1/50/99% = {q[:,0].round(1)} dB | VH = {q[:,1].round(1)} dB")

## 2. Load the five models

In [ ]:
import torch
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
models = []
for path in MODEL_FILES:
    m = smp.Unet(config.ENCODER, encoder_weights=None, in_channels=2, classes=1)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    models.append(m.to(device).eval())
print(len(models), "models loaded")

## 3. Predict over the whole area

The image is cut into 512 x 512 tiles that overlap by 64 pixels. Each tile goes through the **same** `preprocess_sar` used in training, then through all five models. Where tiles overlap, answers are averaged with more weight towards each tile's centre, so there are no seams.

In [ ]:
# Weight that is high in the tile centre and low (but not zero) at the edges
ramp = np.minimum(np.arange(TILE) + 1, np.arange(TILE)[::-1] + 1).astype(np.float32)
ramp = np.minimum(ramp / 64.0, 1.0) * 0.99 + 0.01
WEIGHT = np.outer(ramp, ramp)

def starts(n):
    s = list(range(0, max(n - TILE, 0) + 1, STRIDE))
    if s[-1] + TILE < n:
        s.append(n - TILE)
    return s

@torch.no_grad()
def predict_all(db):
    """Water probability from each model for the whole image: float16 array (5, H, W)."""
    acc = np.zeros((len(models), H, W), dtype=np.float32)
    wsum = np.zeros((H, W), dtype=np.float32)
    boxes = [(r, c) for r in starts(H) for c in starts(W)]
    t0 = time.time()
    for i in range(0, len(boxes), BATCH):
        chunk = boxes[i:i + BATCH]
        x = np.stack([preprocess_sar(db[:, r:r + TILE, c:c + TILE]) for r, c in chunk])
        xb = torch.from_numpy(x).to(device)
        with torch.autocast(device_type="cuda", enabled=device == "cuda"):
            probs = [torch.sigmoid(m(xb)[:, 0].float()).cpu().numpy() for m in models]
        for j, (r, c) in enumerate(chunk):
            for k in range(len(models)):
                acc[k, r:r + TILE, c:c + TILE] += probs[k][j] * WEIGHT
            wsum[r:r + TILE, c:c + TILE] += WEIGHT
        if (i // BATCH) % 25 == 0:
            print(f"  {i + len(chunk)}/{len(boxes)} tiles, {time.time() - t0:.0f} s")
    return (acc / wsum).astype(np.float16)

print("17 July ..."); P_pre = predict_all(pre_db)
print("29 July ..."); P_post = predict_all(post_db)

## 4. Flood rule, once per model

A pixel is flooded for model k only if: water on 29 July, dry on 17 July, JRC occurrence at most 80%, HAND at most 15 m, and both radar images have data there.

In [ ]:
not_permanent = jrc <= config.JRC_PERMANENT_PCT
low_ground = np.nan_to_num(hand_m, nan=0.0) <= config.HAND_MAX_M   # missing HAND: do not exclude
keep = valid & not_permanent & low_ground

flood_bits = np.zeros((H, W), dtype=np.uint8)
water29_bits = np.zeros((H, W), dtype=np.uint8)
for k in range(len(models)):
    w_post = P_post[k] > THRESHOLD
    w_pre = P_pre[k] > THRESHOLD
    flood_bits |= ((w_post & ~w_pre & keep).astype(np.uint8) << k)
    water29_bits |= ((w_post & keep).astype(np.uint8) << k)

def count_bits(b):
    return sum(((b >> k) & 1) for k in range(len(models))).astype(np.uint8)

flood_count = count_bits(flood_bits)
water29_count = count_bits(water29_bits)
print("Done")

## 5. Areas (km2)

In [ ]:
import pandas as pd
PX_KM2 = (10 * 10) / 1e6
rows = []
for k in range(len(models)):
    f = (flood_bits >> k) & 1
    w = (water29_bits >> k) & 1
    rows.append({"map": f"model {k}", "flooded_km2": f.sum() * PX_KM2, "water29_no_rule2_km2": w.sum() * PX_KM2})
for n in (1, 3, 5):
    rows.append({"map": f"at least {n} of 5 agree", "flooded_km2": (flood_count >= n).sum() * PX_KM2,
                 "water29_no_rule2_km2": (water29_count >= n).sum() * PX_KM2})
summary = pd.DataFrame(rows).round(1)

# How much each cleaning rule removed (using the averaged map, for reporting only)
w_post_mean = np.mean(P_post, axis=0, dtype=np.float32) > THRESHOLD
w_pre_mean = np.mean(P_pre, axis=0, dtype=np.float32) > THRESHOLD
removed = {
    "new water before cleaning": (w_post_mean & ~w_pre_mean & valid).sum() * PX_KM2,
    "removed as permanent water (JRC)": (w_post_mean & ~w_pre_mean & valid & ~not_permanent).sum() * PX_KM2,
    "removed as high ground (HAND)": (w_post_mean & ~w_pre_mean & valid & not_permanent & ~low_ground).sum() * PX_KM2,
}
summary.to_csv(f"{OUT}/summary.csv", index=False)
with open(f"{OUT}/cleaning_removed_km2.json", "w") as f:
    json.dump({k: round(float(v), 1) for k, v in removed.items()}, f, indent=1)
print(summary.to_string(index=False))
print()
for k, v in removed.items():
    print(f"{k:35s} {v:8.1f} km2")

## 6. Save the maps

In [ ]:
def save(name, arr, nodata=None):
    p = prof.copy()
    p.update(count=1, dtype=arr.dtype.name, nodata=nodata, compress="deflate", tiled=True,
             blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
    with rasterio.open(f"{OUT}/{name}.tif", "w", **p) as f:
        f.write(arr, 1)

def as_uint8(x, scale):
    out = np.clip(np.round(x * scale), 0, 250).astype(np.uint8)
    out[~valid] = 255
    return out

mean_post = np.mean(P_post, axis=0, dtype=np.float32)
std_post = np.std(P_post, axis=0, dtype=np.float32)
mean_pre = np.mean(P_pre, axis=0, dtype=np.float32)

save("prob_post_mean", as_uint8(mean_post, 250), 255)
save("prob_post_std", as_uint8(std_post, 500), 255)
save("prob_pre_mean", as_uint8(mean_pre, 250), 255)
save("flood_bits", flood_bits)
save("water29_bits", water29_bits)
save("flood_count", flood_count)
with open(f"{OUT}/run_info.json", "w") as f:
    json.dump({"threshold": THRESHOLD, "tile": TILE, "stride": STRIDE,
               "jrc_permanent_pct": config.JRC_PERMANENT_PCT, "hand_max_m": config.HAND_MAX_M,
               "models": [os.path.basename(m) for m in MODEL_FILES],
               "inputs": [os.path.basename(PRE_TIF[0]), os.path.basename(POST_TIF[0]), os.path.basename(AUX_TIF[0])]},
              f, indent=1)
print(sorted(os.listdir(OUT)))

## 7. Quick look

**Check by eye:**
- Flood (red) should sit on fields and low land along the rivers, not on the main river channel (that is permanent water) and not in the hills.
- Uncertainty should be brightest at the edges of flooded areas.
- If red covers towns, hills or forms straight lines and stripes, something is wrong. Stop and tell Claude.

In [ ]:
import matplotlib.pyplot as plt
S = 10   # show every 10th pixel (100 m) to keep the picture small
vv = np.nan_to_num(post_db[0, ::S, ::S], nan=-30)
fc = flood_count[::S, ::S]
sd = std_post[::S, ::S]

fig, ax = plt.subplots(1, 3, figsize=(21, 6))
ax[0].imshow(vv, cmap="gray", vmin=-25, vmax=0); ax[0].set_title("Radar, 29 July (VV)")
ax[1].imshow(vv, cmap="gray", vmin=-25, vmax=0)
ax[1].imshow(np.ma.masked_equal(fc, 0), cmap="autumn_r", vmin=1, vmax=5)
ax[1].set_title("Flooded: colour = how many of 5 models agree (yellow 1, red 5)")
im = ax[2].imshow(np.where(valid[::S, ::S], sd, np.nan), cmap="magma", vmin=0, vmax=0.3)
ax[2].set_title("Uncertainty (spread of the 5 models)"); plt.colorbar(im, ax=ax[2], fraction=0.03)
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/overview.png", dpi=120); plt.show()

In [ ]:
# Close-up at full 10 m detail around Sivasagar town
import pyproj
lat, lon = config.DISTRICTS["Sivasagar"]["town"]
x, y = pyproj.Transformer.from_crs("EPSG:4326", prof["crs"], always_xy=True).transform(lon, lat)
col, row = ~prof["transform"] * (x, y)
r0, c0, n = int(row) - 750, int(col) - 750, 1500
win = (slice(max(r0, 0), max(r0, 0) + n), slice(max(c0, 0), max(c0, 0) + n))

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
ax[0].imshow(np.nan_to_num(pre_db[0][win], nan=-30), cmap="gray", vmin=-25, vmax=0); ax[0].set_title("17 July (before)")
ax[1].imshow(np.nan_to_num(post_db[0][win], nan=-30), cmap="gray", vmin=-25, vmax=0); ax[1].set_title("29 July (during)")
ax[2].imshow(np.nan_to_num(post_db[0][win], nan=-30), cmap="gray", vmin=-25, vmax=0)
ax[2].imshow(np.ma.masked_equal(flood_count[win], 0), cmap="autumn_r", vmin=1, vmax=5, alpha=0.8)
ax[2].set_title("Flood, models agreeing (15 km x 15 km around Sivasagar)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/closeup_sivasagar.png", dpi=110); plt.show()

## Done

**Save Version > Save & Run All** if you ran this interactively, so the outputs are kept. Then download `outputs/` from the *Output* tab and send Claude:
- the printed tables from section 5,
- the two pictures in `figures/`.

Do **not** treat these areas as final: the model has not yet been tested on hand-marked Assam tiles.